# 03 – Globale Temperatur aus Land und Ozean

Wir kombinieren die **Landstationen** (GHCN-Monthly v4, homogenisiert) mit der **Meeresoberflächentemperatur** (ERSST v5) zu einer eigenen globalen Temperaturkurve und vergleichen sie mit **NASA GISTEMP** und **HadCRUT5** (Met Office).

Methode (`src/klima/land_ozean.py`):
1. Ozean: SST-Anomalie je 2°-Zelle gegenüber 1951–1980, Meereis (−1,8 °C) als fehlend, gemittelt auf 5°-Zellen,
2. Land: 5°-Gitteranomalien aus GHCNm wie in Notebook 02,
3. je Zelle: Landanteil × Land + (1 − Landanteil) × Ozean (Landanteil aus Natural Earth),
4. flächengewichtetes Mittel über alle besetzten Zellen.

In [ ]:
%matplotlib inline
import pandas as pd

from klima import auswertungen
from klima import land_ozean as lo
from klima import weltweit as ww
from klima.anomalien import standard_referenzperiode
from klima.grafik import interaktiv, statisch
from klima.trend import linearer_trend

pd.set_option("display.precision", 2)
REFERENZ = standard_referenzperiode()

## Landanteil je 5°-Zelle

In [ ]:
anteil = lo.landanteil(5.0)
statisch.gitterkarte(
    anteil.rename(columns={"landanteil": "anomalie"}).assign(anomalie=lambda t: t.anomalie * 100),
    "Landanteil je 5°-Zelle",
    einheit="%",
    grenze=100,
);

## Land, Ozean und Kombination

In [ ]:
land = ww.gitter_mittel(ww.stationsanomalien("qcf", REFERENZ), groesse=5.0)
ozean = lo.ozeananomalien(REFERENZ, 5.0)
kombiniert = lo.kombinieren(land, ozean, anteil)
jahre = ww.jahresanomalien(kombiniert)
anteile = pd.DataFrame(
    {
        "Land": ww.jahresanomalien(land)["global"],
        "Ozean": ww.jahresanomalien(ozean)["global"],
        "Land + Ozean": jahre["global"],
    }
).loc[1880:]
{name: linearer_trend(anteile[name], von=1951).text() for name in anteile}

In [ ]:
statisch.zeitreihen(
    anteile, "Land, Ozean und Kombination", referenz=REFERENZ, farben=auswertungen.FARBEN_LAND_OZEAN
);

## Vergleich mit GISTEMP und HadCRUT5

HadCRUT5 bezieht sich auf 1961–1990 und wird auf 1951–1980 umgerechnet. Die Variante **„nicht aufgefüllt“** mittelt wie wir nur über Zellen mit Messungen; **„aufgefüllt“** interpoliert statistisch in Lücken.

In [ ]:
letztes = int(jahre.index.max())
vergleich = pd.DataFrame(
    {
        auswertungen.GISTEMP_LO: auswertungen._gistemp_jahre("global", "land_ozean"),
        auswertungen.EIGENE_LO: jahre["global"],
        auswertungen.HADCRUT_NICHT: auswertungen._hadcrut_jahre(
            "global", "nicht_aufgefuellt", REFERENZ
        ),
        auswertungen.HADCRUT_AUF: auswertungen._hadcrut_jahre("global", "aufgefuellt", REFERENZ),
    }
).loc[1880:letztes]
vergleich.corr()

In [ ]:
interaktiv.zeitreihen(
    vergleich,
    "Globale Temperatur Land + Ozean",
    referenz=REFERENZ,
    farben=auswertungen.FARBEN_LAND_OZEAN,
)

In [ ]:
differenz = vergleich.drop(columns=auswertungen.EIGENE_LO).rsub(
    vergleich[auswertungen.EIGENE_LO], axis=0
)
differenz.agg(["mean", "std", "min", "max"])

Die größten Abweichungen zu HadCRUT liegen um 1940 – dort korrigieren ERSST und HadSST4 die Umstellung der Messpraxis auf Schiffen während des Zweiten Weltkriegs unterschiedlich. Seit etwa 1990 liegen wir etwas unter GISTEMP: Wir lassen Meereis aus, GISTEMP überträgt Landwerte auch in die sich stark erwärmende Arktis.

## Anomalie je Jahrzehnt (Zeitregler)

In [ ]:
jahrzehnte = ww.jahrzehntmittel(kombiniert)
interaktiv.gitterkarte_zeitregler(
    jahrzehnte[jahrzehnte.jahrzehnt >= 1880],
    "jahrzehnt",
    "Land und Ozean je Jahrzehnt",
    beschriftung=lambda z: f"{z}er",
)